In [14]:
import cv2
import numpy as np

# Paths
CFG = "yolo_files/yolov3.cfg"
WEIGHTS = "yolo_files/yolov3.weights"
CLASSES = "yolo_files/yolov3.txt"
# IMAGE = 'toy_dataset/nina_on_couch.jpeg'
IMAGE = 'toy_dataset/microglia_big_screenshot.jpg'
IGNORE_CLASS_LABELS = False
CONF_THRESHOLD = 0.1
NMS_THRESHOLD = 0.1


In [18]:

def get_output_layers(net):
    layer_names = net.getLayerNames()
    try:
        output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]
    except:
        output_layers = [layer_names[i[0] - 1] for i in net.getUnconnectedOutLayers()]
    return output_layers


def draw_prediction(img, class_id, confidence, x, y, x_plus_w, y_plus_h):

    if IGNORE_CLASS_LABELS:
        label = f"{confidence:.2f}"
        color = (0, 255, 0)  # green for all boxes
    else:
        label = f"{classes[class_id]}: {confidence:.2f}"
        color = COLORS[class_id]

    cv2.rectangle(img, (x, y), (x_plus_w, y_plus_h), color, 2)
    cv2.putText(
        img,
        label,
        (x - 10, y - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        color,
        2
    )



image = cv2.imread(IMAGE)
Height, Width = image.shape[:2]
scale = 0.00392




with open(CLASSES, 'r') as f:
    classes = [line.strip() for line in f.readlines()]

COLORS = np.random.uniform(0, 255, size=(len(classes), 3))



net = cv2.dnn.readNet(WEIGHTS, CFG)

blob = cv2.dnn.blobFromImage(
    image,
    scale,
    (416, 416),
    (0, 0, 0),
    swapRB=True,
    crop=False
)

net.setInput(blob)
outs = net.forward(get_output_layers(net))



class_ids = []
confidences = []
boxes = []

for out in outs:
    for detection in out:

        objectness = detection[4]
        scores = detection[5:]
        class_id = np.argmax(scores)
        class_score = scores[class_id]

        if IGNORE_CLASS_LABELS:
            confidence = objectness * class_score
        else:
            confidence = class_score

        if confidence > CONF_THRESHOLD:
            center_x = int(detection[0] * Width)
            center_y = int(detection[1] * Height)
            w = int(detection[2] * Width)
            h = int(detection[3] * Height)
            x = int(center_x - w / 2)
            y = int(center_y - h / 2)

            class_ids.append(class_id)
            confidences.append(float(confidence))
            boxes.append([x, y, w, h])


# =========================
# NON-MAX SUPPRESSION
# =========================

indices = cv2.dnn.NMSBoxes(
    boxes,
    confidences,
    CONF_THRESHOLD,
    NMS_THRESHOLD
)

for i in indices:
    try:
        i = i[0]
    except:
        pass

    box = boxes[i]
    x, y, w, h = box

    draw_prediction(
        image,
        class_ids[i],
        confidences[i],
        x,
        y,
        x + w,
        y + h
    )


# =========================
# SHOW & SAVE
# =========================

cv2.imwrite("toy_dataset/object-detection.jpg", image)
print("Saved result to toy_dataset/object-detection.jpg")

Saved result to toy_dataset/object-detection.jpg
